# NBA Stats Data Collection

Pulls team box-score data for the 2025–26 NBA regular season from the NBA stats API and saves the raw tables used by `02_kalshi_data.ipynb` and `03_data_prep.ipynb`. It produces three kinds of data: full-game box scores, rolling averages of each team's previous games (for pregame features), and cumulative in-game box scores at the end of quarters 1–3 (for in-game features).

**Output files** (saved to `data/raw/`, two rows per game, one per team):

| File | Contents |
|---|---|
| `_team-box-stats(full-game).csv` | Full-game box score for every team in every game, including final score, `WL`, and `PLUS_MINUS` |
| `roll-avg-only(3GAME).csv`, `(5GAME)`, `(10GAME)`, `(15GAME)` | Each team's average box-score stats over its previous 3, 5, 10, or 15 games, excluding the current game |
| `in-game-box-stats(EndQ1).csv`, `(EndQ2)`, `(EndQ3)` | Cumulative box-score stats from tip-off through the end of that quarter, with game date, matchup, and final result attached |

**Stats covered:** `FGM`, `FGA`, `FG_PCT`, `FG3M`, `FG3A`, `FG3_PCT`, `FTM`, `FTA`, `FT_PCT`, `OREB`, `DREB`, `REB`, `AST`, `STL`, `BLK`, `TOV`, `PF`, `PTS`, `PLUS_MINUS`

## 1. Setup

Imports libraries, including the `nba_api` endpoints for the season game log and the per-quarter box score, and sets the season used throughout the notebook.

In [1]:
import time
import pandas as pd
from nba_api.stats.endpoints import leaguegamelog, boxscoretraditionalv3

pd.set_option('display.max_columns', None)

SEASON = "2025-26"


## 2. Helper Functions

Functions used by the steps that follow:

- **`team_box_stats_full_game`**: pulls the team-level game log for the season, with one row per team per game.
- **`rolling_avg_team_box_score_stats`**: sorts each team's games by date and averages each stat over the previous `prev_games` games. The current game is shifted out of the window so the features contain no information from the game being predicted. Shooting percentages are recomputed as total makes / total attempts over the window instead of averaging the per-game percentages, and the first games of each team's season are dropped because they don't have enough history yet.
- **`team_box_stats_by_quarter`**: pulls one game's team box score from tip-off through a given quarter. Failed requests are retried up to 5 times with an increasing wait. An `END_Q` column records the last quarter included, derived from minutes played.
- **`pull_in_game_box_stats`**: runs `team_box_stats_by_quarter` for every game with a short pause between requests, and stacks the results.
- **`merge_in_game_stats`**: attaches game date, matchup, and the final win/loss to the in-game rows so they can be joined to the other tables.


In [ ]:
# ------------------------------------------------------------------------- #
######### PULL BOX STATS FOR THE FULL GAME #########
# ------------------------------------------------------------------------- #
def team_box_stats_full_game(years):
    '''
    Pull full-game team box scores for one regular season.
    Queries LeagueGameLog at the team level (not player) for the given
    season and returns one row per team per game.

    Parameters:
        years (str): season in NBA API format "2025-26"
    Returns:
        teams (df): team game log with identifiers stats
    '''
    teams = leaguegamelog.LeagueGameLog(
        season=years,
        season_type_all_star="Regular Season",
        player_or_team_abbreviation="T",
    ).get_data_frames()[0]
    teams = teams.drop(columns=["SEASON_ID", "MIN", "VIDEO_AVAILABLE"])
    return teams



# ------------------------------------------------------------------------- #
######### CONVERT BOX STATS TO ROLLING AVERAGE #########
# ------------------------------------------------------------------------- #
def rolling_avg_team_box_score_stats(years, prev_games=3):
    '''
    Build prior-game rolling averages of team box stats for one season.
    Sorts each team oldest-to-newest, then takes the mean of the previous
    `prev_games` games.
    FG_PCT, FG3_PCT, and FT_PCT are rebuilt from makes / attempts
    over that window, not averaged as percentages

    Parameters:
        years (str): season in NBA API format "2025-26"
        prev_games (int): default 3, the number of prior games in the rolling window.
    Returns:
        teams (df): team game log with rolling-average stat columns. Game identity
        columns (GAME_ID, TEAM_ID, GAME_DATE, MATCHUP, WL, etc.) are kept
        as-is so the frame can be joined back to a single game.
    '''
    teams = leaguegamelog.LeagueGameLog(
        season=years,
        season_type_all_star="Regular Season",
        player_or_team_abbreviation="T",
    ).get_data_frames()[0]

    # sort before averaging or so later games don't get put before newer ones
    teams = teams.sort_values(["TEAM_ID", "GAME_DATE", "GAME_ID"])

    # percentages are rebuilt below, so they are not rolled here
    stats = [
        "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
        "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PTS", "PLUS_MINUS"
    ]

    # use shift(1) to not include the current game (prevent leakage)
    teams[stats] = (
        teams.groupby("TEAM_ID")[stats]
        .transform(lambda s: s.shift(1).rolling(prev_games).mean().round(2)))

    # mean(makes) / mean(attempts) == makes / attempts over the window
    teams["FG_PCT"] = teams["FGM"] / teams["FGA"]
    teams["FG3_PCT"] = teams["FG3M"] / teams["FG3A"]
    teams["FT_PCT"] = teams["FTM"] / teams["FTA"]
    stats = stats + ["FG_PCT", "FG3_PCT", "FT_PCT"]
    teams[stats] = teams[stats].round(2)

    teams = teams.rename(columns={s: f"{prev_games}_game_roll_avg_{s}" for s in stats})
    teams = teams.drop(columns=["MIN", "VIDEO_AVAILABLE"])
    return teams.dropna(subset=[f"{prev_games}_game_roll_avg_{s}" for s in stats])



# ------------------------------------------------------------------------- #
######### PULL BOX STATS BY QUARTER #########
# ------------------------------------------------------------------------- #
def team_box_stats_by_quarter(game_id, start_period, end_period, range_type):
    '''
    Pull team box stats for one game over a period range.
    Uses BoxScoreTraditionalV3 team stats from start_period through
    end_period.

    Parameters:
        game_id (str): NBA game id "0022500001"
        start_period (int): first quarter to include (1 = Q1)
        end_period (int): last quarter to include
        range_type (int): BoxScoreTraditionalV3 range type (use 1 for quarters)
    Returns:
        box (df): one row per team in the game, and box-score stats through
        end_period.
    '''
    box = None
    for attempt in range(5):
        try:
            box = boxscoretraditionalv3.BoxScoreTraditionalV3(
                game_id=game_id,
                start_period=start_period,
                end_period=end_period,
                range_type=range_type,
                start_range=0,
                end_range=0,
                timeout=90,
            ).team_stats.get_data_frame()
            break
        except Exception:
            time.sleep(2 ** attempt)
    if box is None:
        print("skip", game_id)
        return None

    box = box.drop(columns=["teamCity", "teamSlug"])

    # convert column to int, total mins played by players on the team (so divide by 5 here)
    box['minutes'] = box['minutes'].astype(str).str.split(':').str[0].astype(int) / 5
    # then find which quarter by dividing by 12
    box['minutes'] = box['minutes'] / 12

    box = box.rename(columns={
        "gameId": "GAME_ID",
        "teamId": "TEAM_ID",
        "teamTricode": "TEAM_ABBREVIATION",
        "minutes": "END_Q",
        "teamName": "TEAM_NAME",
        "fieldGoalsMade": "FGM",
        "fieldGoalsAttempted": "FGA",
        "fieldGoalsPercentage": "FG_PCT",
        "threePointersMade": "FG3M",
        "threePointersAttempted": "FG3A",
        "threePointersPercentage": "FG3_PCT",
        "freeThrowsMade": "FTM",
        "freeThrowsAttempted": "FTA",
        "freeThrowsPercentage": "FT_PCT",
        "reboundsOffensive": "OREB",
        "reboundsDefensive": "DREB",
        "reboundsTotal": "REB",
        "assists": "AST",
        "steals": "STL",
        "blocks": "BLK",
        "turnovers": "TOV",
        "foulsPersonal": "PF",
        "points": "PTS",
        "plusMinusPoints": "PLUS_MINUS"
    })
    return box



# ------------------------------------------------------------------------- #
######### PULL IN-GAME BOX STATS #########
# ------------------------------------------------------------------------- #
def pull_in_game_box_stats(list_of_game_ids, start_period, end_period, range_type):
    '''
    Pull the same period-range team box for many games.
    Calls team_box_stats_by_quarter once per game id and concatenates the
    results. 

    Parameters: 
        list_of_game_ids (lst of strs): NBA game ids to pull.
        start_period (int): first quarter to include (1 = Q1)
        end_period (int): last quarter to include
        range_type (int): BoxScoreTraditionalV3 range type (use 1 for quarters)
    Returns
        df: stacked team rows for every game id, with the same columns as
        team_box_stats_by_quarter.
    '''
    frames = []
    for game_id in list_of_game_ids:
        frame = team_box_stats_by_quarter(
            game_id, 
            start_period=start_period, 
            end_period=end_period, 
            range_type=range_type
        )
        time.sleep(1) # pause to avoid getting rate limited by nba api
        if frame is not None:
            frames.append(frame)

    df = pd.concat(frames, ignore_index=True)
    return df



# ------------------------------------------------------------------------- #
######### MERGE IN-GAME BOX STATS WITH WL/DATE INFO #########
# ------------------------------------------------------------------------- #
def merge_in_game_stats(df1, df2):
    '''
    Attach game date, matchup, and win/loss to in-game team box rows.
    Left-joins df2 (in-game box stats) to a slim slice of df1 (full-game
    team log) on GAME_ID and TEAM_ID.

    Parameters:
        df1: full-game team log, must have GAME_ID, TEAM_ID, GAME_DATE,
        MATCHUP, and WL.
        df2: in-game team box stats, must have GAME_ID and TEAM_ID.
    Returns:
        df: in-game stats with GAME_DATE, MATCHUP, and WL attached
    '''
    df = df2.merge(
        df1[['GAME_ID', 'TEAM_ID', 'GAME_DATE', 'MATCHUP', 'WL']],
        on=['GAME_ID', 'TEAM_ID'],
        how='left'
    )

    df = df[[
        "GAME_ID", "GAME_DATE", 
        "TEAM_ID", "TEAM_NAME", "TEAM_ABBREVIATION", "MATCHUP", "END_Q",
        "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT", "FTM", "FTA", "FT_PCT",
        "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PTS", "PLUS_MINUS", "WL"        
    ]]
    return df



## 3. Full-Game Box Scores

Pulls the full-game box score for every team in every regular-season game and saves it. This file is the base game list for the rest of the project: `02_kalshi_data.ipynb` uses it to build the one-row-per-game table, and `03_data_prep.ipynb` uses it to check the rolling averages for leakage. Full-game stats are never used directly as model inputs.

In [ ]:
test = team_box_stats_full_game(SEASON)
test.to_csv("../data/raw/_team-box-stats(full-game).csv", index=False)
test.head(4)



,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS
0,1610612744,GSW,Golden State Warriors,0022500002,2025-10-21,GSW @ LAL,W,38,78,0.487,17,40,0.425,26,29,0.897,9,31,40,29,10,4,19,27,119,10
1,1610612745,HOU,Houston Rockets,0022500001,2025-10-21,HOU @ OKC,L,43,97,0.443,11,39,0.282,27,31,0.871,16,36,52,23,6,5,25,26,124,-1
2,1610612747,LAL,Los Angeles Lakers,0022500002,2025-10-21,LAL vs. GSW,L,42,77,0.545,8,32,0.250,17,28,0.607,7,32,39,23,7,2,20,21,109,-10
3,1610612760,OKC,Oklahoma City Thunder,0022500001,2025-10-21,OKC vs. HOU,W,46,104,0.442,13,52,0.250,20,25,0.800,11,27,38,29,12,4,12,27,125,1


## 4. Rolling Averages

Builds pregame features from each team's recent form: the average of each box-score stat over the team's previous games. Four window lengths (3, 5, 10, and 15 games) are saved so they can be compared. `03_data_prep.ipynb` uses the 5-game window.

In [4]:
test1 = rolling_avg_team_box_score_stats(SEASON, prev_games=3)
test1.head(4)


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,3_game_roll_avg_FGM,3_game_roll_avg_FGA,3_game_roll_avg_FG_PCT,3_game_roll_avg_FG3M,3_game_roll_avg_FG3A,3_game_roll_avg_FG3_PCT,3_game_roll_avg_FTM,3_game_roll_avg_FTA,3_game_roll_avg_FT_PCT,3_game_roll_avg_OREB,3_game_roll_avg_DREB,3_game_roll_avg_REB,3_game_roll_avg_AST,3_game_roll_avg_STL,3_game_roll_avg_BLK,3_game_roll_avg_TOV,3_game_roll_avg_PF,3_game_roll_avg_PTS,3_game_roll_avg_PLUS_MINUS
84,22025,1610612737,ATL,Atlanta Hawks,0022500115,2025-10-27,ATL @ CHI,L,38.00,86.67,0.44,11.33,34.33,0.33,22.33,28.67,0.78,10.67,29.33,40.00,25.33,7.67,5.33,16.33,22.67,109.67,-11.00
116,22025,1610612737,ATL,Atlanta Hawks,0022500130,2025-10-29,ATL @ BKN,W,41.67,89.33,0.47,11.67,33.33,0.35,16.33,23.00,0.71,10.67,30.00,40.67,27.67,8.00,3.67,13.00,20.33,111.33,-6.00
144,22025,1610612737,ATL,Atlanta Hawks,0022500020,2025-10-31,ATL @ IND,W,42.67,92.00,0.46,13.33,34.67,0.38,14.67,21.00,0.70,12.00,30.00,42.00,30.00,8.67,4.67,10.33,22.00,113.33,-5.67
172,22025,1610612737,ATL,Atlanta Hawks,0022500151,2025-11-02,ATL @ CLE,L,48.00,95.00,0.51,11.33,31.67,0.36,15.33,20.67,0.74,11.00,34.00,45.00,31.67,9.67,4.67,8.00,22.00,122.67,6.67


In [ ]:
test1.to_csv("../data/raw/roll-avg-only(3GAME).csv", index=False)

test2 = rolling_avg_team_box_score_stats(SEASON, prev_games=5)
test2.to_csv("../data/raw/roll-avg-only(5GAME).csv", index=False)

test3 = rolling_avg_team_box_score_stats(SEASON, prev_games=10)
test3.to_csv("../data/raw/roll-avg-only(10GAME).csv", index=False)

test4 = rolling_avg_team_box_score_stats(SEASON, prev_games=15)
test4.to_csv("../data/raw/roll-avg-only(15GAME).csv", index=False)

## 5. In-Game Box Scores

Collects the game IDs from the full-game table, then pulls each game's cumulative team box score through the end of Q1, Q2, and Q3, which matches the in-game checkpoints. Each pull makes one request per game and takes about 20 minutes. Each result is joined to the full-game table for date, matchup, and final result, then saved per checkpoint.

In [22]:
all_ids = test["GAME_ID"].unique().tolist()
len(all_ids)


1230

In [ ]:
##### Pull for end of Q1 (takes about 20min)
q1_df = pull_in_game_box_stats(all_ids, start_period=1, end_period=1, range_type=1)
q1_df_final = merge_in_game_stats(test, q1_df)
q1_df_final.head(4)


,GAME_ID,GAME_DATE,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,MATCHUP,END_Q,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,WL
0,0022500001,2025-10-21,1610612760,Thunder,OKC,OKC vs. HOU,1.0,10,19,0.526,1,7,0.143,6,7,0.857,2,6,8,7,1,1,5,5,-3.0,W
1,0022500001,2025-10-21,1610612745,Rockets,HOU,HOU @ OKC,1.0,12,23,0.522,2,8,0.250,4,4,1.000,3,6,9,7,5,0,3,5,3.0,L
2,0022500002,2025-10-21,1610612747,Lakers,LAL,LAL vs. GSW,1.0,8,19,0.421,1,9,0.111,5,7,0.714,1,9,10,4,2,1,8,5,-6.0,L
3,0022500002,2025-10-21,1610612744,Warriors,GSW,GSW @ LAL,1.0,8,20,0.400,5,10,0.500,7,7,1.000,1,9,10,6,7,1,6,6,6.0,W


In [ ]:
##### Pull for end of Q2 (takes about 20min)
q2_df = pull_in_game_box_stats(all_ids, start_period=1, end_period=2, range_type=1)
q2_df_final = merge_in_game_stats(test, q2_df)
q2_df_final.head(4)


,GAME_ID,GAME_DATE,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,MATCHUP,END_Q,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,WL
0,0022500001,2025-10-21,1610612760,Thunder,OKC,OKC vs. HOU,2.0,19,43,0.442,5,20,0.250,8,9,0.889,5,11,16,14,3,2,7,10,-6.0,W
1,0022500001,2025-10-21,1610612745,Rockets,HOU,HOU @ OKC,2.0,20,43,0.465,7,19,0.368,10,10,1.000,7,15,22,13,5,1,8,10,6.0,L
2,0022500002,2025-10-21,1610612747,Lakers,LAL,LAL vs. GSW,2.0,20,36,0.556,5,17,0.294,9,16,0.563,3,18,21,10,4,2,14,13,-1.0,L
3,0022500002,2025-10-21,1610612744,Warriors,GSW,GSW @ LAL,2.0,16,40,0.400,7,18,0.389,16,18,0.889,6,15,21,13,8,1,13,14,1.0,W


In [ ]:
##### Pull for end of Q3 (takes about 20min)
q3_df = pull_in_game_box_stats(all_ids, start_period=1, end_period=3, range_type=1)
q3_df_final = merge_in_game_stats(test, q3_df)
q3_df_final.head(4)


,GAME_ID,GAME_DATE,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,MATCHUP,END_Q,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,WL
0,0022500001,2025-10-21,1610612760,Thunder,OKC,OKC vs. HOU,3.0,29,67,0.433,8,33,0.242,9,11,0.818,7,16,23,19,8,2,9,16,-4.0,W
1,0022500001,2025-10-21,1610612745,Rockets,HOU,HOU @ OKC,3.0,27,63,0.429,8,28,0.286,17,17,1.000,13,24,37,15,6,2,15,13,4.0,L
2,0022500002,2025-10-21,1610612747,Lakers,LAL,LAL vs. GSW,3.0,30,56,0.536,6,24,0.250,13,23,0.565,5,24,29,15,5,2,17,17,-11.0,L
3,0022500002,2025-10-21,1610612744,Warriors,GSW,GSW @ LAL,3.0,28,60,0.467,12,28,0.429,22,24,0.917,8,25,33,22,9,1,15,20,11.0,W


In [ ]:
q1_df_final.to_csv("../data/raw/in-game-box-stats(EndQ1).csv", index=False)
q2_df_final.to_csv("../data/raw/in-game-box-stats(EndQ2).csv", index=False)
q3_df_final.to_csv("../data/raw/in-game-box-stats(EndQ3).csv", index=False)
